# 14. A Complete Numerical WalkthroughFrom *Transformers from Scratch*.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or# three files the chapter uses. If you cloned the repository it# finds them where they already are and downloads nothing.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"PLACES = [".", "code", "../code"]for name in ["ch14_attention.py", "ch14_pytorch.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

One attention head, computed with numbers you can check.Four tokens, a model dimension of four, one head, a key dimension of four. Every weight here is a nought or a one and every embedding is a small whole number, so every product below can be checked with a pencil. The scores divide by the square root of four, which is exactly two.

In [ ]:
import numpy as npnp.set_printoptions(precision=3, suppress=True, floatmode="fixed")TOKENS = ["the", "cat", "drank", "milk"]# One row per token. In a trained model these are learned. Here they are# chosen so that "cat" and "milk" share a feature and "drank" does not.X = np.array([    [1.0, 0.0, 1.0, 0.0],   # the    [0.0, 2.0, 0.0, 1.0],   # cat    [1.0, 0.0, 2.0, 0.0],   # drank    [0.0, 1.0, 0.0, 2.0],   # milk])W_Q = np.array([    [0.0, 0.0, 1.0, 0.0],    [1.0, 1.0, 0.0, 0.0],    [0.0, 0.0, 0.0, 1.0],    [0.0, 1.0, 0.0, 0.0],])W_K = np.array([    [1.0, 0.0, 0.0, 0.0],    [0.0, 1.0, 1.0, 0.0],    [0.0, 1.0, 0.0, 0.0],    [0.0, 0.0, 0.0, 1.0],])W_V = np.array([    [1.0, 0.0, 0.0, 0.0],    [0.0, 1.0, 0.0, 1.0],    [0.0, 0.0, 1.0, 0.0],    [1.0, 0.0, 0.0, 1.0],])

Row-wise softmax, shifted by the row max so it cannot overflow.

In [ ]:
def softmax(scores):    shifted = scores - scores.max(axis=-1, keepdims=True)    exponentiated = np.exp(shifted)    return exponentiated / exponentiated.sum(axis=-1, keepdims=True)

One head of scaled dot-product attention, keeping every step.

In [ ]:
def attention(x, w_q, w_k, w_v):    queries = x @ w_q    keys = x @ w_k    values = x @ w_v    d_k = keys.shape[-1]    raw = queries @ keys.T    scaled = raw / np.sqrt(d_k)    weights = softmax(scaled)    output = weights @ values    return queries, keys, values, raw, scaled, weights, output

Print one labelled matrix the way the book prints it.

In [ ]:
def show(name, matrix, cols):    print(f"\n{name}   shape {matrix.shape}")    print(" " * 7 + "".join(f"{c:>8}" for c in cols))    for label, row in zip(TOKENS, matrix):        print(f"{label:>6} " + "".join(f"{v:>8.3f}" for v in row))

In [ ]:
def main():    q, k, v, raw, scaled, weights, out = attention(X, W_Q, W_K, W_V)    show("X, the input embeddings", X, ["e0", "e1", "e2", "e3"])    show("Q = X W_Q", q, ["q0", "q1", "q2", "q3"])    show("K = X W_K", k, ["k0", "k1", "k2", "k3"])    show("V = X W_V", v, ["v0", "v1", "v2", "v3"])    show("Raw scores, Q K-transpose", raw, TOKENS)    show("Scaled scores, every entry divided by 2", scaled, TOKENS)    show("Attention weights, softmax along each row", weights, TOKENS)    show("Output, weights times V", out, ["o0", "o1", "o2", "o3"])    print("\nRow sums of the attention weights:", weights.sum(axis=1))

In [ ]:
main()

Now the second half of the chapter.Chapter 14, section 7. The same head in PyTorch, and the proof.The point of this file is the two assertions at the bottom. The framework is doing the arithmetic from ch14_attention.py and nothing else, and the only way to believe that is to run both and compare the numbers.

In [ ]:
import numpy as npimport torchimport torch.nn.functional as Ffrom ch14_attention import X, W_Q, W_K, W_V, attentiontorch.set_printoptions(precision=3, sci_mode=False)

One head, written with PyTorch operations instead of NumPy.

In [ ]:
def attention_torch(x, w_q, w_k, w_v):    queries = x @ w_q    keys = x @ w_k    values = x @ w_v    d_k = keys.shape[-1]    scores = queries @ keys.transpose(-2, -1) / d_k ** 0.5    weights = torch.softmax(scores, dim=-1)    return weights, weights @ values

In [ ]:
def main():    x = torch.tensor(X, dtype=torch.float64)    w_q = torch.tensor(W_Q, dtype=torch.float64)    w_k = torch.tensor(W_K, dtype=torch.float64)    w_v = torch.tensor(W_V, dtype=torch.float64)    weights, output = attention_torch(x, w_q, w_k, w_v)    print("Attention weights from PyTorch")    print(weights)    print("\nOutput from PyTorch")    print(output)    # the same call, handed to the built-in fused implementation    builtin = F.scaled_dot_product_attention(x @ w_q, x @ w_k, x @ w_v)    _, _, _, _, _, np_weights, np_output = attention(X, W_Q, W_K, W_V)    assert np.allclose(weights.numpy(), np_weights, atol=1e-12)    assert np.allclose(output.numpy(), np_output, atol=1e-12)    assert np.allclose(builtin.numpy(), np_output, atol=1e-12)    print("\nNumPy, PyTorch and F.scaled_dot_product_attention agree")    print("largest disagreement:",          float(np.abs(builtin.numpy() - np_output).max()))

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.